# Step 8 — Validate FinBERT on the reviewed headlines

We score only the headline text with `ProsusAI/finbert`, a model trained for financial sentiment classification. Its outputs remain separate from the dataset-provided scores.

In [1]:
from pathlib import Path

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "ProsusAI/finbert"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REVIEW_PATH = PROJECT_ROOT / "data" / "processed" / "apple_sentiment_manual_review.csv"
review = pd.read_csv(REVIEW_PATH)
assert review["manual_sentiment"].notna().all()

/Users/keishakalra/Desktop/Financial_App/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Score the headlines

The tokenizer converts each headline into model inputs. Softmax converts the model's raw outputs into three probabilities that sum to one. The highest probability determines the predicted label.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
model.config.id2label

{0: 'positive', 1: 'negative', 2: 'neutral'}

In [3]:
encoded = tokenizer(
    review["title"].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt",
)
with torch.no_grad():
    probabilities = torch.softmax(model(**encoded).logits, dim=1).cpu().numpy()

id_to_label = {int(key): value.lower() for key, value in model.config.id2label.items()}
for class_id, label in id_to_label.items():
    review[f"finbert_{label}_probability"] = probabilities[:, class_id]

winning_ids = probabilities.argmax(axis=1)
review["finbert_label"] = [id_to_label[int(class_id)].title() for class_id in winning_ids]
review["finbert_confidence"] = probabilities.max(axis=1)
review.to_csv(REVIEW_PATH, index=False)
review[["review_id", "title", "manual_sentiment", "finbert_label", "finbert_confidence"]].head(10)

,review_id,title,manual_sentiment,finbert_label,finbert_confidence
0,1,More than $1 trillion wiped off value of Apple...,Negative,Negative,0.927494
1,2,"Stocks open lower, Tesla stock climbs as Apple...",Neutral,Negative,0.951017
2,3,This Is Retail Investors' Favorite Stock to Ow...,Neutral,Neutral,0.946270
3,4,Beyond the iPhone: Here's What May Decide Appl...,Neutral,Neutral,0.938912
4,5,Shares of Apple suppliers fall on reports of C...,Negative,Negative,0.964522
5,6,"iPhone 15 launch: Release date, price and new ...",Neutral,Neutral,0.945337
6,7,Apple's Superpower? It's About the Ecosystem.,Positive,Neutral,0.932009
7,8,Apple Inc. (NASDAQ:AAPL) is largely controlled...,Neutral,Neutral,0.957112
8,9,The company that makes your iPhone is expandin...,Neutral,Neutral,0.681074
9,10,Apple Unveils M3 Processors and New MacBook Pros,Neutral,Neutral,0.557463


## Compare with human judgments

Accuracy is exact label agreement. Cohen's kappa adjusts agreement for what might occur by chance. We report the dataset score and FinBERT on the same 30 headlines.

In [4]:
comparison = pd.DataFrame({
    "Dataset polarity": {
        "Accuracy": accuracy_score(review["manual_sentiment"], review["dataset_label"]),
        "Cohen's kappa": cohen_kappa_score(review["manual_sentiment"], review["dataset_label"]),
    },
    "FinBERT headline": {
        "Accuracy": accuracy_score(review["manual_sentiment"], review["finbert_label"]),
        "Cohen's kappa": cohen_kappa_score(review["manual_sentiment"], review["finbert_label"]),
    },
}).T
comparison.round(3)

,Accuracy,Cohen's kappa
Dataset polarity,0.400,0.217
FinBERT headline,0.767,0.565


In [5]:
labels = ["Negative", "Neutral", "Positive"]
pd.DataFrame(
    confusion_matrix(review["manual_sentiment"], review["finbert_label"], labels=labels),
    index=[f"Manual: {label}" for label in labels],
    columns=[f"FinBERT: {label}" for label in labels],
)

,FinBERT: Negative,FinBERT: Neutral,FinBERT: Positive
Manual: Negative,7,1,0
Manual: Neutral,2,15,0
Manual: Positive,0,4,1


In [6]:
review.loc[review["manual_sentiment"].ne(review["finbert_label"]), [
    "review_id", "title", "manual_sentiment", "finbert_label", "finbert_confidence"
]].sort_values("finbert_confidence", ascending=False)

,review_id,title,manual_sentiment,finbert_label,finbert_confidence
1,2,"Stocks open lower, Tesla stock climbs as Apple...",Neutral,Negative,0.951017
6,7,Apple's Superpower? It's About the Ecosystem.,Positive,Neutral,0.932009
13,14,"Walgreens, Ford, Micron rise premarket; Apple,...",Negative,Neutral,0.922064
25,26,Is Apple Inc (NASDAQ:AAPL) the Best AI Tech St...,Positive,Neutral,0.907342
23,24,Famous Analyst Thinks Apple Inc (NASDAQ:AAPL) ...,Positive,Neutral,0.793140
29,30,"Market Chatter: Google, Apple Asked by Indones...",Neutral,Negative,0.652931
18,19,Apple Buys Canadian AI Startup as It Races to ...,Positive,Neutral,0.498656


## Limitations

Thirty examples provide a useful quality check but not a definitive benchmark. Our labels are subjective, and one reviewer confirmed them. FinBERT also evaluates linguistic tone—not whether a headline will move the stock.